In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import numpy as np

class TradingVisualizer:
    def __init__(self, env):
        self.env = env
        self.current_step = env.lookback_window
        
    def render_step(self, step=None):
        """Render a specific step"""
        if step is None:
            step = self.current_step
        else:
            self.current_step = step
            
        # Get data for current lookback window
        start_idx = max(0, step - self.env.lookback_window)
        end_idx = start_idx + self.env.lookback_window  # +1 to include current step
        window_data = self.env.data.iloc[start_idx:end_idx]
        
    
        # Create the chart
        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.7, 0.3],
            shared_yaxes=True,
            horizontal_spacing=0.02,
            subplot_titles=('Price Action', 'Volume Profile')
        )
        
        # 1. Candlestick chart (left)
        fig.add_trace(
            go.Candlestick(
                x=window_data.index,
                open=window_data['open'],
                high=window_data['high'],
                low=window_data['low'],
                close=window_data['close'],
                name="Price"
            ),
            row=1, col=1
        )

        fig.update_xaxes(title_text="Time", row=1, col=1)
        fig.update_xaxes(title_text="Volume Density", row=1, col=2)
        fig.update_yaxes(title_text="Price", row=1, col=1)
        
        return fig.show()
    
 

# Usage:
def add_visualizer_to_env(env):
    """Add visualizer to your existing environment"""
    visualizer = TradingVisualizer(env)
    
    # Add methods to env
    env.visualizer = visualizer
    env.render_step = visualizer.render_step
    
    return visualizer

In [8]:

# Configuration
import os

import pandas as pd

from environments.simple_trading_env import SimpleTradingEnv


# === CONFIGURATION ===
DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
MODEL_PATH = "trading_bot"
LOOKBACK_WINDOW = 1000

def create_env_for_dataset(data):
    """Create environment for specific dataset"""
    return SimpleTradingEnv(
        data=data,
        initial_balance=10000,
        lookback_window=LOOKBACK_WINDOW,  # Adjust based on your data
        # Add other env parameters you need
    )

# Train individual models with SPARSE REWARD optimization
models = []
training_logs = []



# Load data
df = pd.read_pickle(DATA_PATH)
env = create_env_for_dataset(df)

# After running your environment
visualizer = add_visualizer_to_env(env)

# Method 3: Use the visualizer directly
visualizer.render_step(0)

Info: Dropped 99 rows due to NaNs after adding indicators.
